# 04_finetune_local_embedder.ipynb

Fine-tune the local embedder (all-MiniLM-L6-v2) on a self-constructed finance corpus (template-generated sentence pairs; one variant also uses FiQA). The report describes one epoch of fine-tuning (Sec. 6.1). The last training cell runs that variant and saves it as `models/financial-st-mini-less-1`, which notebook 07 loads.

Paths are relative: `data/` holds dialogues and embeddings (not included), `models/` holds fine-tuned weights (not included). See the README.


## Optional variant: template pairs + FiQA (not used for the reported results)

In [ ]:
import torch
import random
from datasets import Dataset, load_dataset, concatenate_datasets
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)

fiqa = load_dataset("BeIR/fiqa-generated-queries", split="train")

In [ ]:
import random
from sentence_transformers import InputExample

train_examples = []

amounts = ["50000", "80000", "120000", "200000", "1000", "2400", "3500", "800", "200", "340", "5260", "4387", "3498", "2311", "23", "34", "56"]
percentages = [10, 30, 50, 70, 90, 40, 20, 80, 60, 12.5, 37.2, 69.3, 74.5, 23.3, 63.9, 23.6, 12.1, 89.7]

income_templates = [
    ("salary is {}", "annual income is {}"),
    ("he earns {} yearly", "his income is {}"),
    ("compensation is {}", "total earnings are {}")
]

investment_templates = [
    ("portfolio allocation is {}% stocks", "investment is {} percent equities"),
    ("{}% of assets are bonds", "bond allocation is {}%"),
    ("fund is {}% equity heavy", "equity exposure is {}%")
]

bill_templates = [
    ("monthly bill is {}", "monthly expense is {}"),
    ("utility bill costs {}", "utility expense is {}"),
    ("credit card bill is {}", "credit card spending is {}"),
    ("rent bill is {}", "housing expense is {}"),
]

bill_hard_negatives = [
    ("monthly bill is {}", "monthly income is {}"),
    ("utility expense is {}", "utility income is {}"),
    ("credit card bill is {}", "credit card reward is {}"),
    ("rent expense is {}", "rental income is {}"),
]

In [ ]:
for _ in range(3000):
    amt = random.choice(amounts)
    p = random.choice(percentages)

    t1, t2 = random.choice(income_templates)
    train_examples.append(InputExample(texts=[t1.format(amt), t2.format(amt)]))

    t1, t2 = random.choice(investment_templates)
    train_examples.append(InputExample(texts=[t1.format(p), t2.format(p)]))

    b = random.choice(amounts)
    t1, t2 = random.choice(bill_templates)
    train_examples.append(InputExample(texts=[t1.format(b), t2.format(b)]))

for _ in range(2000):
    a1 = random.choice(amounts)
    a2 = random.choice(amounts)

    if a1 == a2:
        continue

    train_examples.append(
        InputExample(texts=[
            ("salary is {}".format(a1)),
            ("monthly income is {}".format(a2))
        ])
    )

    t1, t2 = random.choice(bill_hard_negatives)
    train_examples.append(
        InputExample(texts=[t1.format(a1), t2.format(a2)])
    )

for item in fiqa:
    query = item["query"]
    passage = item["text"]

    if query is None or passage is None:
        continue

    train_examples.append(
        InputExample(texts=[query, passage])
    )

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=64
)

In [ ]:
from sentence_transformers import losses

train_loss = losses.MultipleNegativesRankingLoss(model)

In [ ]:
# Optional variant (also uses FiQA); not used for the reported results.
model.fit(
    train_objectives=[(train_dataloader, train_loss)],

    epochs=3,

    warmup_steps=int(len(train_dataloader) * 4 * 0.1),

    optimizer_params={
        "lr": 2e-5,
        "weight_decay": 0.01
    },
    scheduler="WarmupLinear",
    use_amp=True,
    output_path="models/financial-st-mini-query",
    show_progress_bar=True,
    checkpoint_save_steps=1000,
    checkpoint_save_total_limit=2
)

model.max_seq_length = 256

## Reported variant: template-generated finance pairs

In [ ]:
from sentence_transformers import InputExample

income_templates = [
    "My salary is {} dollars per year",
    "I earn {} annually",
    "My income is {} per month",
    "I make about {} a year"
]

bill_templates = [
    "My monthly bill is {} dollars",
    "I spend {} on expenses",
    "My utility bill is {}",
    "I pay {} for rent"
]

investment_templates = [
    "I invest {} percent of my income",
    "My portfolio allocation is {}% stocks",
    "I put {}% into bonds",
    "My investment share is {} percent"
]

values = ["50000", "80000", "120000", "200000", "1000", "2400", "3500", "800", "200", "340", "5260", "4387", "3498", "2311", "23", "34", "56"]
percents = [10, 30, 50, 70, 90, 40, 20, 80, 60, 12.5, 37.2, 69.3, 74.5, 23.3, 63.9, 23.6, 12.1, 89.7]

train_examples = []

for v1 in values:
    for v2 in values:
        for t1 in income_templates:
            for t2 in income_templates:
                train_examples.append(
                    InputExample(
                        texts=[t1.format(v1), t2.format(v2)],
                        label=1.0
                    )
                )
for v1 in values:
    for v2 in values:
        for t1 in bill_templates:
            for t2 in bill_templates:
                train_examples.append(
                    InputExample(
                        texts=[t1.format(v1), t2.format(v2)],
                        label=1.0
                    )
                )
for p1 in percents:
    for p2 in percents:
        for t1 in investment_templates:
            for t2 in investment_templates:
                train_examples.append(
                    InputExample(
                        texts=[t1.format(p1), t2.format(p2)],
                        label=1.0
                    )
                )

for v in values:
    for t1 in income_templates:
        for t2 in bill_templates:
            train_examples.append(
                InputExample(
                    texts=[t1.format(v), t2.format(v)],
                    label=0.0
                )
            )

for v in values:
    for t1 in income_templates:
        for t2 in investment_templates:
            train_examples.append(
                InputExample(
                    texts=[t1.format(v), t2.format("40")],
                    label=0.0
                )
            )

for v in values:
    for t1 in bill_templates:
        for t2 in investment_templates:
            train_examples.append(
                InputExample(
                    texts=[t1.format(v), t2.format("40")],
                    label=0.0
                )
            )

In [ ]:
income_templates = [
    "My salary is {} dollars per year",
    "I earn {} annually",
    "My income is {} per month",
    "I make about {} a year",
    "My yearly earnings are {}",
    "I bring in {} annually",
    "My monthly income is {}",
    "I get paid {} per year",
    "My take-home pay is {}",
    "I earn roughly {} each year"
]

bill_templates = [
    "My monthly bill is {} dollars",
    "I spend {} on expenses",
    "My utility bill is {}",
    "I pay {} for rent",
    "My monthly expenses are {}",
    "I usually spend {} per month",
    "My bills total {} monthly",
    "I have to pay {} every month",
    "My rent costs {}",
    "My total expenses come to {}"
]

investment_templates = [
    "I invest {} percent of my income",
    "My portfolio allocation is {}% stocks",
    "I put {}% into bonds",
    "My investment share is {} percent",
    "I allocate {}% of my income to investments",
    "About {}% goes into my portfolio",
    "I invest roughly {}% each month",
    "My asset allocation is {}%",
    "I dedicate {}% to stocks",
    "I keep {}% in investments"
]

import random

In [ ]:
def sample_pairs(templates, values, n=3000):
    pairs = []
    for _ in range(n):
        t1, t2 = random.sample(templates, 2)
        v1 = random.choice(values)
        v2 = random.choice(values)
        pairs.append((t1.format(v1), t2.format(v2)))
    return pairs

In [ ]:
train_examples = []

for s1, s2 in sample_pairs(income_templates, values):
    train_examples.append(InputExample(texts=[s1, s2], label=1.0))

for s1, s2 in sample_pairs(bill_templates, values):
    train_examples.append(InputExample(texts=[s1, s2], label=1.0))

for s1, s2 in sample_pairs(investment_templates, percents):
    train_examples.append(InputExample(texts=[s1, s2], label=1.0))

In [ ]:
def cross_pairs(tpl1, tpl2, vals1, vals2, n=3000):
    pairs = []
    for _ in range(n):
        t1 = random.choice(tpl1)
        t2 = random.choice(tpl2)
        v1 = random.choice(vals1)
        v2 = random.choice(vals2)
        pairs.append((t1.format(v1), t2.format(v2)))
    return pairs

In [ ]:
for s1, s2 in cross_pairs(income_templates, bill_templates, values, values):
    train_examples.append(InputExample(texts=[s1, s2], label=0.0))

for s1, s2 in cross_pairs(income_templates, investment_templates, values, percents):
    train_examples.append(InputExample(texts=[s1, s2], label=0.0))

for s1, s2 in cross_pairs(bill_templates, investment_templates, values, percents):
    train_examples.append(InputExample(texts=[s1, s2], label=0.0))

In [ ]:
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=64)

train_loss = losses.CosineSimilarityLoss(model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],

    epochs=1,

    warmup_steps=int(len(train_dataloader) * 4 * 0.1),

    optimizer_params={
        "lr": 2e-5,
        "weight_decay": 0.01
    },
    scheduler="WarmupLinear",
    use_amp=True,
    output_path="models/financial-st-mini-less-1",
    show_progress_bar=True,
    checkpoint_save_steps=1000,
    checkpoint_save_total_limit=2
)